## Feature Engineering
**Goal:** turn raw Instacart orders into a modelling table with one row per (user, product) pair, holding features about that pair and a label saying whether the user bought the product in their next order.

**Inputs:** 'orders.csv', 'order_products__prior.csv', 'order_products__train.csv' from 'data/raw/'

**Outputs:** 'data/processed/model_dataset.parquet', with features, labels and a train/ validation/test split column.

**Leakage rule:** every feature is built **only from prior orders**. The target order (eval_set='train') is used only to create the label.

## 1. Setup
Paths to the raw and processed folders, and in-memory DuckDB connection. DuckDB reads the CSVs directly, so the 32M-row prior file never has to be loaded fully into pandas.    

In [2]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd().parent
RAW_DATA_DIR = PROJECT_DIR/"data"/"raw"
PROCESSED_DATA_DIR = PROJECT_DIR/"data"/"processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

orders_path = RAW_DATA_DIR/"orders.csv"
prior_path = RAW_DATA_DIR/"order_products__prior.csv"
train_path = RAW_DATA_DIR/"order_products__train.csv"
products_path = RAW_DATA_DIR/"products.csv"

con = duckdb.connect()

print("Feature engineering environment initialized!")
print("Raw data directory exists: ", RAW_DATA_DIR.exists())
print("Processed data directory: ", PROCESSED_DATA_DIR)
print("Orders file exists: ", orders_path.exists())
print("Prior products file exists: ", prior_path.exists())
print("Training products file exists: ", train_path.exists())

Feature engineering environment initialized!
Raw data directory exists:  True
Processed data directory:  c:\grocery_reorder_prediction\data\processed
Orders file exists:  True
Prior products file exists:  True
Training products file exists:  True


### Target orders at a glance
The train file holds each training user's **last order**, the order we are trying to predict: 1,384,617 product rows across 131,209 orders (one per user). 

In [3]:
train_summary = con.execute("""
    SELECT
    COUNT(*) AS product_order_rows,
    COUNT(DISTINCT order_id) AS unique_orders,
    COUNT(DISTINCT product_id) AS unique_products
    FROM read_csv_auto(?)
""", [str(train_path)]).df()

display(train_summary)

,product_order_rows,unique_orders,unique_products
0,1384617,131209,39123


## 2. User-product history features

**Candidates:** for each user, every product they bought in at least one prior order.

| Feature | Meaning |
|---|---|
|purchase_count| Number of prior orders containing the product |
|purchase_share| purchase_count/user's total prior orders: how often this product appears in the user's orders (0-1) |
|orders_since_last_purchase| Orders since the user last bought it (0=bought in their most recent order) |

reorder_ratio is also calculated here so it can be checked in the next cell.

In [5]:
historical_features = con.execute("""
    WITH user_orders AS(
        SELECT order_id, user_id, order_number 
        FROM read_csv_auto(?)
        WHERE eval_set = 'prior'
    ),

    product_history AS(
        SELECT o.user_id, op.product_id, COUNT(*) AS purchase_count, 
            SUM(op.reordered) AS reorder_count,
            MAX(o.order_number) AS last_order_number
        FROM read_csv_auto(?) AS op
        JOIN user_orders AS o
            ON op.order_id = o.order_id
        GROUP BY
            o.user_id,
            op.product_id
    ),

    user_summary AS (
        SELECT user_id, MAX(order_number) AS last_prior_order_number
        FROM user_orders
        GROUP BY user_id
    )

    SELECT h.user_id, h.product_id, h.purchase_count,
    h.reorder_count * 1.0/ h.purchase_count AS reorder_ratio,
    h.purchase_count * 1.0/ u.last_prior_order_number AS purchase_share,
    u.last_prior_order_number - h.last_order_number AS orders_since_last_purchase
    FROM product_history AS h
    JOIN user_summary AS u ON h.user_id = u.user_id
""", [str(orders_path), str(prior_path)]).df()

display(historical_features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,user_id,product_id,purchase_count,reorder_ratio,purchase_share,orders_since_last_purchase
0,107145,46969,2,0.500000,0.117647,3
1,107145,23288,6,0.833333,0.352941,3
2,160785,8021,1,0.000000,0.333333,0
3,197183,23989,1,0.000000,0.076923,0
4,129462,1445,4,0.750000,0.057143,13


### Is reorder-ratio a useful feature?
reorder_ratio = reorder_count/purchase_count. The **first** purchase of a product is never a reorder, so for a product bought *n* times , reorder_count = n-1. That means reorder_ratio should always equal (n-1)/n, which is just purchase_count written differently.

In [6]:
diff = (historical_features["reorder_ratio"] - (historical_features["purchase_count"]-1)/historical_features["purchase_count"]).abs()
print("Total user_product pairs: ", len(historical_features))
print("Rows where reorder_ratio differs from (n-1)/n:", (diff > 1e-9).sum())

Total user_product pairs:  13307953
Rows where reorder_ratio differs from (n-1)/n: 0


**Result:** reorder_ratio equals (n-1)/n in **all 13,307,953 rows** (0 differ). It adds no information beyond purchase_count, so it is dropped.

It is replaced by purchase_share, which brings in new information: the user's total order count. For example, a product bought 10 times in 10 orders (share 1.0) is a staple. The same 10 purchases spread over 99 orders (share 0.10) is occasional. reorder_ratio would give both 0.9 .

In [7]:
historical_features = historical_features.drop(columns=["reorder_ratio"])
historical_features.head()

,user_id,product_id,purchase_count,purchase_share,orders_since_last_purchase
0,107145,46969,2,0.117647,3
1,107145,23288,6,0.352941,3
2,160785,8021,1,0.333333,0
3,197183,23989,1,0.076923,0
4,129462,1445,4,0.057143,13


### Sanity check
13,307,953 user-product pairs across 206,209 users and 49,677 products. User 1 (10 prior orders) shows the features behaving as expected: products bought in every order have purchase_share = 1.0 and orders_since_last_purchase=0.

In [8]:
print("Number of user-product pairs: ", len(historical_features))
print("Number of users: ", historical_features["user_id"].nunique())
print("Number of products: ", historical_features["product_id"].nunique())

display(historical_features.sort_values(["user_id", "purchase_count"], ascending=[True, False]).head(10))

Number of user-product pairs:  13307953
Number of users:  206209
Number of products:  49677


,user_id,product_id,purchase_count,purchase_share,orders_since_last_purchase
1803262,1,12427,10,1.0,0
8779760,1,196,10,1.0,0
2026390,1,10258,9,0.9,0
1803261,1,25133,8,0.8,0
229133,1,46149,3,0.3,0
395494,1,13032,3,0.3,0
306887,1,49235,2,0.2,1
522175,1,26088,2,0.2,8
2630125,1,13176,2,0.2,5
10312106,1,26405,2,0.2,6


## 3. Labels for training users
Only 131,209 of the 206,209 users have a known next order (eval_set=train). The rest are Kaggle's test users, whose answers are not public, so they are dropped here.

**Label = 1** if the candidate product appears in the user's next order, otherwise **0**. Products in the next order that the user never bought before cannot be candidates. That is the natural ceiling of a "Buy it again" row.

Saved to `train_features.parquet`.

### Does due-ness predict reorders
Reorder rate by `due_ratio` bucket (overall rate is 9.78%):
|due_ratio|Meaning|Reorder rate|
|---|---|---|
|no gap|bought only once|4.7%|
|0-0.5|bought too recently|20.7%|
|0.5-1|nearly due|26.9%|
|1-1.5|just due|25.4%| 
|1.5-3|overdue|17.3%|
|3+|long overdue (likely dropped)|6.6%|

The rate peaks when a product is about due (ratio around 1) and falls on both sides.That is a clear signal, visible before training any model.

In [9]:
con.register("historical_features_df", historical_features)

train_features_path = (PROCESSED_DATA_DIR/"train_features.parquet")

orders_file = orders_path.as_posix()
train_file = train_path.as_posix()
output_file = train_features_path.as_posix()

con.execute(f"""
    COPY(
        WITH training_users AS (
            SELECT user_id
            FROM read_csv_auto('{orders_file}')
            WHERE eval_set = 'train'
        ),
        target_products AS (
            SELECT o.user_id, op.product_id
            FROM read_csv_auto('{train_file}') AS op
            JOIN read_csv_auto('{orders_file}') AS o
            ON op.order_id = o.order_id
            WHERE o.eval_set = 'train'
        )

        SELECT h.user_id, h.product_id, h.purchase_count, h.purchase_share, h.orders_since_last_purchase,
        CASE   
            WHEN t.product_id IS NOT NULL THEN 1
            ELSE 0
        END AS label
        FROM historical_features_df AS h
        JOIN training_users AS u
        ON h.user_id = u.user_id
        LEFT JOIN target_products AS t
        ON h.user_id = t.user_id AND h.product_id = t.product_id 
    )
    TO '{output_file}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Training features saved to:", train_features_path)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training features saved to: c:\grocery_reorder_prediction\data\processed\train_features.parquet


In [10]:
con.unregister("historical_features_df")
del historical_features
import gc; gc.collect()
print("historical_features freed from memory")

historical_features freed from memory


### Label checks

In [11]:
train_validation = con.execute(f"""
    SELECT
        COUNT(*) AS total_candidate_pairs,
        COUNT(DISTINCT user_id) AS unique_users,
        COUNT(DISTINCT product_id) AS unique_products,
        CAST(SUM(label) AS BIGINT) AS positive_labels,
        AVG(label) AS positive_label_rate
    FROM read_parquet('{output_file}')
""").df()

display(train_validation)

,total_candidate_pairs,unique_users,unique_products,positive_labels,positive_label_rate
0,8474661,131209,49468,828824,0.0978


**Positive rate is 9.78%:** about 1 in 10 candidates is reordered, so the classes are imbalanced. The distribution below shows how many reordered products each user has.

In [12]:
label_distribution = con.execute(f"""
    SELECT labels_per_user, COUNT(*) AS number_of_users
    FROM (
        SELECT user_id, SUM(label) AS labels_per_user
        FROM read_parquet('{output_file}')
        GROUP BY user_id
    )
    GROUP BY labels_per_user
    ORDER BY labels_per_user
""").df()

display(label_distribution)

,labels_per_user,number_of_users
0,0.0,8602
1,1.0,15498
2,2.0,14773
3,3.0,13281
4,4.0,12014
...,...,...
58,60.0,1
59,61.0,2
60,63.0,1
61,65.0,1


**8,602 users (6.6%) have zero positives:** their next order contained only products they had never bought before. No "buy it again" row could help them, so they are excluded when calculating recall later (recall is undefined with nothing to find).

In [13]:
coverage_summary = con.execute(f"""
    WITH user_labels AS (
        SELECT user_id, SUM(label) AS positive_products
        FROM read_parquet('{output_file}')
        GROUP BY user_id
    )
    SELECT COUNT(*) AS total_users,
    SUM(
        CASE WHEN positive_products>0
        THEN 1 ELSE 0 END
    ) AS users_with_positive_labels,
    SUM(
        CASE WHEN positive_products=0
        THEN 1 ELSE 0 END
    ) AS users_with_zero_labels,
    ROUND(
        100.0*SUM(
            CASE WHEN positive_products>0
            THEN 1 ELSE 0 END
        )/COUNT(*),2) AS positive_user_coverage_pct
    FROM user_labels
""").df()

display(coverage_summary)

,total_users,users_with_positive_labels,users_with_zero_labels,positive_user_coverage_pct
0,131209,122607.0,8602.0,93.44


### Schema and class balance

In [14]:
feature_schema = con.execute(f"""
    DESCRIBE SELECT(*) FROM read_parquet('{output_file}')
""").df()

display(feature_schema)

,column_name,column_type,null,key,default,extra
0,user_id,BIGINT,YES,None,None,None
1,product_id,BIGINT,YES,None,None,None
2,purchase_count,BIGINT,YES,None,None,None
3,purchase_share,DOUBLE,YES,None,None,None
4,orders_since_last_purchase,BIGINT,YES,None,None,None
5,label,INTEGER,YES,None,None,None


In [15]:
feature_summary = con.execute(f"""
    SELECT COUNT(*) AS total_rows, SUM(label) AS positive_rows,
    COUNT(*) - SUM(label) AS negative_rows, 
    ROUND(AVG(label)*100,2) AS positive_rate_pct
    FROM read_parquet('{output_file}')
""").df()

display(feature_summary)

,total_rows,positive_rows,negative_rows,positive_rate_pct
0,8474661,828824.0,7645837.0,9.78


### Average gap between orders
Across all prior orders, users order roughly every **10.7 days**. This is used as a reference for the per-user timing feature in the next section. 
Note: Instacart caps `days_since_prior_orders` at 30.

In [16]:
user_timing_check = con.execute(f"""
    SELECT AVG(TRY_CAST(days_since_prior_order AS DOUBLE))AS avg_days_between_orders
    FROM read_csv_auto('{orders_file}')
    WHERE eval_set = 'prior'
""").df()

display(user_timing_check)

,avg_days_between_orders
0,10.707242


## 4. User-level and product-level features

| Feature | Level | Meaning |
|---|---|---|
|`user_total_prior_orders`|User|How long the user's history is|
|`user_avg_days_between_orders`|User|How often they shop (a user's first order has no gap and is ignored by `AVG`)|
|`product_total_purchases`|Product|Overall popularity|
|`product_reorder_rate`|Product|Share of the product's purchases that were reorders, i.e. how "habit-forming" it is (milk vs. a one-off spice)|

All are built from prior orders only. `product_reorder_rate` is fine here even though `reorder_ratio` was not: it is calculated across **all users** of a product, so it does not collapse into one user's purchase count.

saved to `train_features_enhanced.parquet`

In [17]:
enhanced_path = (PROCESSED_DATA_DIR/"train_features_enhanced.parquet")
output_file = train_features_path.as_posix()
enhanced_file = enhanced_path.as_posix()
orders_file = orders_path.as_posix()
prior_file = prior_path.as_posix()

con.execute(f"""
    COPY(
        WITH user_stats AS(
            SELECT user_id, COUNT(*) AS user_total_prior_orders,
            AVG(TRY_CAST(days_since_prior_order AS DOUBLE)) AS user_avg_days_between_orders
            FROM read_csv_auto('{orders_file}')
            WHERE eval_set = 'prior'
            GROUP BY user_id 
            ),

        product_stats AS (
            SELECT product_id, COUNT(*) AS product_total_purchases, 
            AVG(reordered) AS product_reorder_rate
            FROM read_csv_auto('{prior_file}')
            GROUP BY product_id
        )

        SELECT f.*, u.user_total_prior_orders, u.user_avg_days_between_orders,
        p.product_total_purchases, p.product_reorder_rate
        FROM read_parquet('{output_file}') AS f
        JOIN user_stats AS u
        ON f.user_id = u.user_id
        JOIN product_stats AS p
        ON f.product_id = p.product_id
        )
        TO '{enhanced_file}'
        (FORMAT PARQUET, COMPRESSION ZSTD, OVERWRITE_OR_IGNORE TRUE)
""")

print("Updated features dataset saved: ", enhanced_path)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Updated features dataset saved:  c:\grocery_reorder_prediction\data\processed\train_features_enhanced.parquet


In [18]:
enhanced_summary = con.execute(f"""
    SELECT COUNT(*) AS total_rows, COUNT(DISTINCT(user_id)) AS unique_users,
    SUM(label) AS positive_labels,
    AVG(label) AS positive_rate,
    AVG(user_total_prior_orders) AS avg_user_orders,
    AVG(product_total_purchases) AS avg_product_popularity
    FROM read_parquet('{enhanced_file}')
""").df()

display(enhanced_summary)

,total_rows,unique_users,positive_labels,positive_rate,avg_user_orders,avg_product_popularity
0,8474661,131209,828824.0,0.0978,25.389131,22510.753825


In [19]:
enhanced_schema = con.execute(f"""
    DESCRIBE SELECT(*) FROM read_parquet('{enhanced_file}')
""").df()

display(enhanced_schema)

,column_name,column_type,null,key,default,extra
0,user_id,BIGINT,YES,None,None,None
1,product_id,BIGINT,YES,None,None,None
2,purchase_count,BIGINT,YES,None,None,None
3,purchase_share,DOUBLE,YES,None,None,None
4,orders_since_last_purchase,BIGINT,YES,None,None,None
5,label,INTEGER,YES,None,None,None
6,user_total_prior_orders,BIGINT,YES,None,None,None
7,user_avg_days_between_orders,DOUBLE,YES,None,None,None
8,product_total_purchases,BIGINT,YES,None,None,None
9,product_reorder_rate,DOUBLE,YES,None,None,None


## 5. Due-ness features
Rebuild each user's day timeline from `days_since_prior_order` (running_sum), then for each (user, product): days since last purchase, usual days between purchases and **due_ratio** = days since/usual gap. Built from prior orders only; the target order's day is known at prediction time.

In [25]:
orders_path = RAW_DATA_DIR/"orders.csv"
orders_file = orders_path.as_posix()

con.execute(f"""
    CREATE OR REPLACE TABLE order_timeline AS
    SELECT order_id, user_id, eval_set, order_number, 
    TRY_CAST(days_since_prior_order AS DOUBLE) AS days_since_prior_order,
    SUM(COALESCE(TRY_CAST(days_since_prior_order AS DOUBLE), 0)) OVER (
        PARTITION BY user_id
        ORDER BY order_number
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS order_day
    FROM read_csv_auto('{orders_file}')
    WHERE eval_set IN ('prior', 'train')
""")

print("order_timetable table created")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

order_timetable table created


In [26]:
display(con.execute("""
    SELECT user_id, order_number, eval_set, days_since_prior_order, order_day
    FROM order_timeline
    WHERE user_id = 1
    ORDER BY order_number
""").df())

timeline_check = con.execute(f"""
    SELECT COUNT(*) AS total_orders,
    COUNT(DISTINCT user_id) AS users,
    COUNT(*) FILTER (WHERE eval_set='train') AS target_orders,
    COUNT(*) FILTER (WHERE order_number=1 AND order_day<>0) AS first_orders_not_day_0,
    MAX(order_day) AS max_day
    FROM order_timeline
""").df()
display(timeline_check)

,user_id,order_number,eval_set,days_since_prior_order,order_day
0,1,1,prior,NaN,0.0
1,1,2,prior,15.0,15.0
2,1,3,prior,21.0,36.0
3,1,4,prior,29.0,65.0
4,1,5,prior,28.0,93.0
5,1,6,prior,19.0,112.0
6,1,7,prior,20.0,132.0
7,1,8,prior,14.0,146.0
8,1,9,prior,0.0,146.0
9,1,10,prior,30.0,176.0


,total_orders,users,target_orders,first_orders_not_day_0,max_day
0,3346083,206209,131209,0,365.0


In [27]:
prior_path = RAW_DATA_DIR/"order_products__prior.csv"
prior_file = prior_path.as_posix()

con.execute(f"""
    CREATE OR REPLACE TABLE due_features AS
    WITH target_orders AS (
        SELECT user_id, order_day AS target_day
        FROM order_timeline
        WHERE eval_set='train'
    ),
    product_days AS (
        SELECT t.user_id, op.product_id,
        MIN(t.order_day) AS first_purchase_day,
        MAX(t.order_day) AS last_purchase_day,
        COUNT(*) AS n_purchases
        FROM read_csv_auto('{prior_file}') AS op
        JOIN order_timeline as t
        ON op.order_id=t.order_id
        WHERE t.eval_set='prior'
        GROUP BY t.user_id, op.product_id
    ),
    gaps AS (
        SELECT p.user_id, p.product_id, 
        g.target_day-p.last_purchase_day AS days_since_last_purchase,
        CASE
            WHEN p.n_purchases>1
            THEN (p.last_purchase_day-p.first_purchase_day)/(p.n_purchases-1)
            END AS avg_days_between_purchases
        FROM product_days AS p
        JOIN target_orders AS g
        ON p.user_id = g.user_id
    )
    SELECT user_id, product_id, days_since_last_purchase, avg_days_between_purchases,
    days_since_last_purchase/NULLIF(avg_days_between_purchases, 0) AS due_ratio
    FROM gaps
""")

print("due_features table created")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

due_features table created


In [28]:
train_features_path = (PROCESSED_DATA_DIR/"train_features.parquet")
output_file = train_features_path.as_posix()

due_check = con.execute(f"""
    SELECT COUNT(*) AS pairs,
    COUNT(*) FILTER (WHERE avg_days_between_purchases IS NULL) AS bought_once_or_same_day,
    MIN(days_since_last_purchase) AS min_days_since,
    MAX(days_since_last_purchase) AS max_days_since,
    MEDIAN(due_ratio) AS median_due_ratio
    FROM due_features 
""").df()
display(due_check)

due_signal = con.execute(f"""
    SELECT CASE
        WHEN d.due_ratio IS NULL THEN 'no gap (bought once)'
        WHEN d.due_ratio<0.5 THEN '0-0.5'
        WHEN d.due_ratio<1 THEN '0.5-1'
        WHEN d.due_ratio<1.5 THEN '1-1.5'
        WHEN d.due_ratio<3 THEN '1.5-3'
        ELSE '3+'
        END AS due_bucket,
        COUNT(*) AS pairs,
        ROUND(AVG(f.label)*100,2) AS reorder_rate_pct
    FROM due_features AS d
    JOIN read_parquet('{output_file}') AS f
    ON d.user_id = f.user_id AND d.product_id = f.product_id
    GROUP BY due_bucket
    ORDER BY MIN(COALESCE(d.due_ratio, -1))
""").df()
display(due_signal)

,pairs,bought_once_or_same_day,min_days_since,max_days_since,median_due_ratio
0,8474661,5083521,0.0,365.0,1.416667


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,due_bucket,pairs,reorder_rate_pct
0,no gap (bought once),5112625,4.71
1,0-0.5,712965,20.73
2,0.5-1,599867,26.87
3,1-1.5,418819,25.39
4,1.5-3,611303,17.26
5,3+,1019082,6.57


## 6. Data quality checks
**Missing values:** none in any feature column

In [20]:
missing_summary = con.execute(f"""
    SELECT COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE purchase_count IS NULL) AS missing_purchase_count,
    COUNT(*) FILTER (WHERE purchase_share IS NULL) AS missing_purchase_share,
    COUNT(*) FILTER (WHERE orders_since_last_purchase IS NULL) AS missing_recency,
    COUNT(*) FILTER (WHERE user_total_prior_orders IS NULL) AS missing_user_orders,
    COUNT(*) FILTER (WHERE user_avg_days_between_orders IS NULL) AS missing_user_timing,
    COUNT(*) FILTER (WHERE product_total_purchases IS NULL) AS missing_product_popularity,
    COUNT(*) FILTER(WHERE product_reorder_rate IS NULL) AS missing_product_reorder_rate
    FROM read_parquet('{enhanced_file}')
""").df()

display(missing_summary)

,total_rows,missing_purchase_count,missing_purchase_share,missing_recency,missing_user_orders,missing_user_timing,missing_product_popularity,missing_product_reorder_rate
0,8474661,0,0,0,0,0,0,0


**Value ranges are all plausible:** `purchase_count` 1-99 , `purchase_share` 0.01-1.0, `orders_since_last_purchase` 0-98, `product_reorder_rate` 0-0.94.

In [21]:
feature_ranges = con.execute(f"""
    SELECT MIN(purchase_count) AS min_purchase_count,
    MAX(purchase_count) AS max_purchase_count,
    MIN(purchase_share) AS min_purchase_share,
    MAX(purchase_share) AS max_purchase_share,
    MIN(orders_since_last_purchase) AS min_recency,
    MAX(orders_since_last_purchase) AS max_recency,
    MIN(product_reorder_rate) AS min_product_reorder_rate,
    MAX(product_reorder_rate) AS max_product_reorder_rate
    FROM read_parquet('{enhanced_file}')
""").df()

display(feature_ranges)

,min_purchase_count,max_purchase_count,min_purchase_share,max_purchase_share,min_recency,max_recency,min_product_reorder_rate,max_product_reorder_rate
0,1,99,0.010101,1.0,0,98,0.0,0.941176


## 7. Train/validation/test split

Users are split **70/15/15** using `HASH(user_id)%100`. Splitting by **user**, not by row, means no user's products appear in two splits, so the model is always tested on users it has never seen. The hash makes the split the same on every run.

In [30]:
con.execute(f"""
    COPY(
        SELECT e.*,
        d.days_since_last_purchase,
        d.avg_days_between_purchases,
        d.due_ratio,
        CASE 
            WHEN HASH(e.user_id)%100 < 70 THEN 'train'
            WHEN HASH(e.user_id)%100 < 85 THEN 'validation'
            ELSE 'test'
        END AS split
        FROM read_parquet('{enhanced_file}') as e
        LEFT JOIN due_features AS d
        ON e.user_id = d.user_id AND e.product_id = d.product_id
    )
    TO '{PROCESSED_DATA_DIR.as_posix()}/model_dataset.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Model dataset created successfully.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Model dataset created successfully.


### Split summary
Positive rates are nearly identical across splits (9.66-9.81%), so the split is balanced.

In [33]:
model_dataset_path = (
    PROCESSED_DATA_DIR/"model_dataset.parquet"
).as_posix()

split_summary = con.execute(f"""
    SELECT split, COUNT(*) AS candidate_pairs, 
    COUNT(DISTINCT user_id) AS unique_users,
    SUM(label) AS positive_labels,
    ROUND(AVG(label)*100,2) AS positive_rate_pct
    FROM read_parquet('{model_dataset_path}')
    GROUP BY split
    ORDER BY split
""").df()

display(split_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,split,candidate_pairs,unique_users,positive_labels,positive_rate_pct
0,test,1297732,19879,125329.0,9.66
1,train,5905950,91546,579248.0,9.81
2,validation,1270979,19784,124247.0,9.78


`due_ratio` and `avg_days_between_purchases` are NULL for 60.3% of rows(products bought only once, so no gap exists). This is expected; LightGBM handles NULLs natively. `days_since_last_purchase` is available for every row

In [32]:
display(con.execute(f"""
    SELECT COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE days_since_last_purchase IS NULL) AS missing_days_since,
    ROUND(AVG(CASE WHEN due_ratio IS NULL THEN 1 ELSE 0 END)*100,1) AS due_ratio_null_pct
    FROM read_parquet('{model_dataset_path}')
""").df())

,total_rows,missing_days_since,due_ratio_null_pct
0,8474661,0,60.3


## 8. Baseline: "most frequently bought"
The simplest rule a real app would ship first: show the user's 10 most-bought products. Ties are broken by most recent purchase (`orders_since_last_purchase ASC`), then `product_id`.

**Recall@k** = (reordered_products in the top k)/(all reordered products for that user), averaged over users. Users with zero positives are excluded.

Any model must clearly beat this number to be worth using.

In [24]:
baseline_results = con.execute(f"""
    WITH ranked AS (
        SELECT user_id, product_id, label,
        ROW_NUMBER() OVER(
            PARTITION BY user_id
            ORDER BY 
                purchase_count DESC,
                orders_since_last_purchase ASC,
                product_id
        ) AS rank
        FROM read_parquet('{model_dataset_path}')
        WHERE split = 'validation'
    ),
    user_metrics AS (
        SELECT  user_id, SUM(label) AS actual_products,
        SUM(
            CASE WHEN rank<=5 THEN label ELSE 0 END
        ) AS hits_at_5,
        SUM(
            CASE WHEN rank<=10 THEN label ELSE 0 END
        ) AS hits_at_10
        FROM ranked 
        GROUP BY user_id
    )

    SELECT COUNT(*) AS validation_users, 
    AVG(hits_at_5 * 1.0/NULLIF(actual_products, 0)) AS recall_at_5,
    AVG(hits_at_10 * 1.0/NULLIF(actual_products, 0)) AS recall_at_10
    FROM user_metrics
    WHERE actual_products>0
""").df()

display(baseline_results)

,validation_users,recall_at_5,recall_at_10
0,18455,0.374197,0.537378


## Summary
|Validation result (18,455 users)|Value|
|---|---|
|Baseline recall@5|0.374|
|Baseline recall@10|0.537|


**Key decisions**
- `reorder_ratio` dropped after proving it equal `(n-1)/n` in every row; replaced with `purchase_share`.
- Using recency as the tie-breaker instead of `reorder_ratio` raised baseline recall@10 from 0.520 to 0.541 (on the earlier 80/20 split).
- 70/15/15 user-level split; test set kept untouched.
- due-ness feature added: reorder rate peaks at ~27% when `due_ratio`is 0.5-1.5 vs 9.8% overall and drops to 6.6% when long overdue (3+).

**Next (`03_modeling.ipynb`):** train LightGBM with and without 3 due-ness features and measure the difference